# Parallactic angle for DDF

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from scipy import stats

from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u
from astropy.timeseries import TimeSeries
from astropy.coordinates import get_sun


from astroplan import Observer
from astroplan import FixedTarget
from astroplan.plots import plot_airmass, plot_parallactic, plot_altitude, plot_sky
from astroplan import is_observable

from pytz import timezone

warnings.filterwarnings("ignore")
print(f"pandas   version : {pd.__version__}")
print(f"numpy    version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
def zenith_tangent_vector(ra_deg, dec_deg, obstime, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith

    Parameters:
    ==========
        ra_deg,dec_deg: target coordinates in the sky
        obstimes:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
    """

    # Source
    sky = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg)

    # Zénith en AltAz → (alt=90°, az arbitraire)
    zenith_altaz = SkyCoord(alt=90 * u.deg, az=0 * u.deg, frame=AltAz(obstime=obstime, location=location))

    # Convertir en ICRS
    zenith_icrs = zenith_altaz.transform_to("icrs")

    # Vecteurs cartésiens
    s = sky.cartesian.xyz.value
    z = zenith_icrs.cartesian.xyz.value

    # Projection tangentielle
    v = z - np.dot(z, s) * s

    # Normalisation
    v /= np.linalg.norm(v)

    return v  # vecteur 3D tangent au ciel

In [ ]:
def zenith_tangent_vector_fromHA(HA_deg, coords, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith
    Parameters:
    ==========
        HA_deg : array of Hour angles
        coords: target SkyCoords
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
        array if sinz values (related to dipole intensity)
    """

    lat_deg = location.lat.to(u.deg).value

    dec_deg = coords.dec.to(u.deg).value
    ra_deg = coords.ra.to(u.deg).value

    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)

    # --- direction source ---
    s = np.array([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])  # (3,)

    # --- zénith ---
    HA_val = HA_deg.to(u.deg).value  # ← FIX unités
    lst = np.deg2rad(HA_val + ra_deg)
    lat = np.deg2rad(lat_deg)

    z = np.array(
        [np.cos(lat) * np.cos(lst), np.cos(lat) * np.sin(lst), np.sin(lat) * np.ones_like(lst)]
    )  # (3, N)

    # --- projection ---
    # v = z - np.dot(z, s) * s
    proj = np.sum(z * s[:, None], axis=0)  # (N,)
    v = z - proj * s[:, None]  # (3, N)

    # --- norme par point ---
    norm = np.linalg.norm(v, axis=0)  # (N,)

    # --- normalisation optionnelle ---
    v_unit = np.zeros_like(v)
    mask = norm > 0
    v_unit[:, mask] = v[:, mask] / norm[mask]

    return v_unit, norm

On note :

- H : angle horaire
- δ : déclinaison de la source
- ϕ : latitude de l’observatoire
- z : angle zénithal
### 1) Cosinus de l’angle zénithal

$$\cos z = \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H$$

### 2) Donc sin(z)
$$\sin z = \sqrt{1 - \left(\sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H\right)^2}$$	

✔ Version compacte équivalente (souvent plus utile)

On passe par l’altitude h :

$$\sin h = \sin\phi\sin\delta + \cos\phi\cos\delta\cos H$$

et :

$$z = \frac{\pi}{2} - h$$

donc :

$$\sin z = \cos h$$

✔ Formule finale (celle de ta fonction Python)

$$\boxed{ \sin z(H,\delta,\phi) = \sqrt{ 1 - \left( \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H \right)^2 } }$$

✔ Interprétation physique (très important pour ton dipôle)
H → rotation du ciel
ϕ → géométrie de l’observatoire
δ → position du champ

👉 donc :

- amplitude du dipôle ∝ sinz(H)
- orientation du dipôle ∝ parallactic angle q(H)

In [ ]:
def sinz_vs_HA(HA_deg, coords, location):
    """
    Compute the sinus of zenith angle from the formula
    \sin z(H,\delta,\phi) = \sqrt{ 1 - \left( \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H \right)^2

    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree

    """

    # --- location ---> latitude
    lat_deg = location.lat.to(u.deg).value

    # --- object ---> declination
    dec_deg = coords.dec.to(u.deg).value

    # --- HA be sure to have quanitites in deg
    HA_valdeg = HA_deg.to(u.deg).value

    HA = np.deg2rad(HA_valdeg)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)

    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    return np.sqrt(1 - cosz**2)

In [ ]:
# -----------------------------
# My computation of  parallactic angle
# -----------------------------
def calculate_parallactic_angle(coords, times, location):
    """
    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree
    """

    # LST
    lst = times.sidereal_time("apparent", longitude=location.lon)

    # angle horaire H = LST - RA
    H = (lst - coords.ra).to(u.rad).value

    # latitude
    phi = location.lat.to(u.rad).value

    # déclinaison
    dec_rad = coords.dec.to(u.rad).value

    # formule du parallactic angle
    sinH = np.sin(H)
    cosH = np.cos(H)

    tan_phi = np.tan(phi)

    num = sinH
    den = tan_phi * np.cos(dec_rad) - np.sin(dec_rad) * cosH

    q = np.arctan2(num, den)

    return np.degrees(q)

In [ ]:
# -----------------------------
# My computation of  parallactic angle
# -----------------------------
def calculate_parallactic_angle_fromHA(ha, coords, location):
    """
    Parameters:
    ==========
        coords: target SkyCoords
        ha:  hour angle Angle in degree
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree
    """

    # angle horaire H = LST - RA
    ha_rad = ha.to(u.rad).value

    # latitude
    phi = location.lat.to(u.rad).value

    # déclinaison
    dec_rad = coords.dec.to(u.rad).value

    # formule du parallactic angle
    sinH = np.sin(ha_rad)
    cosH = np.cos(ha_rad)

    tan_phi = np.tan(phi)

    num = sinH
    den = tan_phi * np.cos(dec_rad) - np.sin(dec_rad) * cosH

    q = np.arctan2(num, den)

    return np.degrees(q)

## Initialisation

### Target initialisations

In [ ]:
# ── LSST Deep Drilling Fields ─────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

DEEP_FIELDS_COLORSTYLE = {
    "COSMOS": {"color": "r"},
    "ELAIS-S1": {"color": "k"},
    "ECDFS": {"color": "grey"},
    "EDFS-a": {"color": "b"},
    "EDFS-b": {"color": "g"},
    "EDFS": {"color": "magenta"},
    "M49": {"color": "purple"},
}

### Observatory initialisation

In [ ]:
# ── Rubin/LSST observatory location (Cerro Pachón) ───────────────────────────
RUBIN_LAT_DEG = -30.244728  # degrees North
RUBIN_LON_DEG = -70.749417  # degrees East  (West is negative)
RUBIN_HEIGHT_M = 2647.0  # metres above sea level

RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Rubin/LSST : lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

### observer in astroplan in case one want to check

In [ ]:
observer = Observer.at_site("lsst", timezone="UTC")

In [ ]:
observer

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")

HA = np.arange(-180.0, 180.0) * u.deg

for key, value in DEEP_FIELDS.items():
    field_name = key
    ra = value[0]
    dec = value[1]
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    label = field_name + " : ( $\delta = $" + f"{dec:.1f} deg )"

    q = calculate_parallactic_angle_fromHA(HA, coords, RUBIN_LOCATION)

    ax.plot(HA, q, label=label, lw=2)

ax.set_xlabel("Hour angle (degrees)")
ax.set_ylabel("Parallactic angle (degrees)")


# --- Axe secondaire en heures ---
def deg2hour(x):
    return x / 15.0


def hour2deg(x):
    return x * 15.0


secax = ax.secondary_xaxis("top", functions=(deg2hour, hour2deg))
secax.set_xlabel("Hour angle (hours)")


ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.01, 1))

ax.set_title(f"LSST Deep Fields parallactic angle vs hour angle")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")

HA = np.arange(-180.0, 180.0) * u.deg

for key, value in DEEP_FIELDS.items():
    field_name = key
    ra = value[0]
    dec = value[1]
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    label = field_name + " : ( $\delta = $" + f"{dec:.1f} deg )"

    # projection of zenith in object tangeant plane
    v, sinz = zenith_tangent_vector_fromHA(HA, coords, RUBIN_LOCATION)

    ax.plot(HA, sinz, label=label, lw=2)

ax.set_xlabel("Hour angle (degrees)")
ax.set_ylabel("sinz")


# --- Axe secondaire en heures ---
def deg2hour(x):
    return x / 15.0


def hour2deg(x):
    return x * 15.0


secax = ax.secondary_xaxis("top", functions=(deg2hour, hour2deg))
secax.set_xlabel("Hour angle (hours)")

ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"LSST Deep Fields sinz vs hour angle")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")

HA = np.arange(-180.0, 180.0) * u.deg

for key, value in DEEP_FIELDS.items():
    field_name = key
    ra = value[0]
    dec = value[1]
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    label = field_name + " : ( $\delta = $" + f"{dec:.1f} deg )"

    # projection of zenith in object tangeant plane
    sinz = sinz_vs_HA(HA, coords, RUBIN_LOCATION)

    ax.plot(HA, sinz, label=label, lw=2)

ax.set_xlabel("Hour angle (degrees)")
ax.set_ylabel("sinz")


# --- Axe secondaire en heures ---
def deg2hour(x):
    return x / 15.0


def hour2deg(x):
    return x * 15.0


secax = ax.secondary_xaxis("top", functions=(deg2hour, hour2deg))
secax.set_xlabel("Hour angle (hours)")

ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"LSST Deep Fields sinz vs hour angle")

plt.show()

### Parallactic angle

In [ ]:
selected_field_name = "COSMOS"
coordinates = SkyCoord(
    DEEP_FIELDS[selected_field_name][0] * u.deg, DEEP_FIELDS[selected_field_name][1] * u.deg, frame="icrs"
)
field_target = FixedTarget(name=selected_field_name, coord=coordinates)

In [ ]:
# 1. Définir début et fin
t_start = Time("2026-01-01 00:00:00")
t_end = Time("2026-01-01 23:59:59")

# 2. Construire grille temporelle (pas = 1 heure)
n_hours = int((t_end - t_start).to(u.hour).value)

times_day = t_start + np.arange(n_hours) * u.hour

In [ ]:
q = calculate_parallactic_angle(coordinates, times_day, RUBIN_LOCATION)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(times_day.to_datetime(), q)
# =========================
# 5. Axe temps propre
# =========================

# locator = mdates.AutoDateLocator()
# formatter = mdates.ConciseDateFormatter(locator)
# ax.xaxis.set_major_locator(locator)
# ax.xaxis.set_major_formatter(formatter)

ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d:%H:%M"))
ax.xaxis.set_minor_locator(mdates.MinuteLocator(interval=15))
fig.autofmt_xdate()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_parallactic(
    field_target,
    observer,
    Time("2026-01-01T00:00:00"),
    ax=ax,
)

ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d:%H:%M"))
ax.xaxis.set_minor_locator(mdates.MinuteLocator(interval=15))
fig.autofmt_xdate()


ax.set_title("Astroplan parallactic angle calculation")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

plot_airmass(
    field_target,
    observer,
    Time("2026-01-01"),
    ax=ax,
    brightness_shading=True,  # nuit/jour
    altitude_yaxis=True,  # altitude en plus
)

locator = mdates.AutoDateLocator()
formatter = mdates.ConciseDateFormatter(locator)

ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(formatter)

ax.legend(shadow=True, loc=2)

plt.show()

In [ ]:
# 1. Définir début et fin
t_start = Time("2026-01-01 00:00:00")
t_end = Time("2026-06-30 23:59:59")

# 2. Construire grille temporelle (pas = 1 heure)
n_hours = int((t_end - t_start).to(u.hour).value)

times_6months = t_start + np.arange(n_hours) * u.hour

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

plot_airmass(field_target, observer, times_6months, ax=ax, brightness_shading=True)

# Formatter personnalisé
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d:%H"))

# Optionnel : espacer les ticks (sinon trop dense)
ax.xaxis.set_major_locator(mdates.AutoDateLocator())

ax.set_ylim(2.25, 1)

# Rotation pour lisibilité
plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

In [ ]:
# =========================
# 1. Time grid (6 mois, 1h)
# =========================
t_start = Time("2026-01-01 00:00:00")
t_end = Time("2026-06-30 23:00:00")

n_hours = int((t_end - t_start).to(u.hour).value)
times = t_start + np.arange(n_hours) * u.hour


# =========================
# 2. Figure setup
# =========================
fig, ax = plt.subplots(figsize=(14, 6), dpi=120)


# =========================
# 3. Plot airmass
# =========================
plot_airmass(
    field_target,
    observer,
    times,
    ax=ax,
    brightness_shading=True,  # nuit/jour
    altitude_yaxis=True,  # altitude à droite
)


# =========================
# 4. Contraintes Rubin
# =========================
# airmass < 1.5
airmass_max = 2.2
ax.axhline(airmass_max, ls="--", lw=2, color="red", alpha=0.7)
ax.text(times[0].datetime, airmass_max, f"Rubin limit (X={airmass_max:.1f})", color="red")


# =========================
# 5. Axe temps propre
# =========================
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d"))

ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))

fig.autofmt_xdate()


# =========================
# 6. Labels
# =========================
ax.set_title("COSMOS visibility (Jan–Jun 2026)")
ax.set_ylabel("Airmass")


# =========================
# 7. Layout
# =========================
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 1. Time grid (6 mois, 1h)
# =========================
t_start = Time("2026-01-01 00:00:00")
t_end = Time("2026-06-30 23:00:00")

n_hours = int((t_end - t_start).to(u.hour).value)
times = t_start + np.arange(n_hours) * u.hour


# =========================
# 2. Coordonnées AltAz
# =========================
altaz_frame = observer.altaz(times, target=field_target)

alt = altaz_frame.alt
airmass = altaz_frame.secz


# =========================
# 3. Masque nuit (astronomique)
# =========================
sun_alt = observer.altaz(times, get_sun(times)).alt
night_mask = sun_alt < -18 * u.deg


# =========================
# 4. Préparer grille (date × heure)
# =========================
# heure locale
local_times = times.to_datetime(timezone=observer.timezone)

hours = np.array([t.hour + t.minute / 60 for t in local_times])
dates = np.array([t.date() for t in local_times])

unique_dates = np.unique(dates)
n_days = len(unique_dates)

# grille vide
grid = np.full((24, n_days), np.nan)

# remplissage
for i, (d, h, am, is_night) in enumerate(zip(dates, hours, airmass, night_mask)):
    if is_night and am > 0:  # visible + nuit
        day_idx = np.where(unique_dates == d)[0][0]
        hour_idx = int(h)
        grid[hour_idx, day_idx] = am

# directement les fenêtres exploitables
# grid[grid > airmass_max] = np.nan


# =========================
# 5. Plot heatmap
# =========================
fig, ax = plt.subplots(figsize=(10, 4))

im = ax.imshow(
    grid,
    origin="lower",
    aspect="auto",
    vmin=1,
    vmax=2,
)

# axe x = dates
ax.set_xticks(np.arange(0, n_days, 14))
ax.set_xticklabels([str(d) for d in unique_dates[::14]], rotation=45)

# axe y = heures
ax.set_yticks(np.arange(0, 24, 2))
ax.set_ylabel("Local hour")

# colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Airmass")

ax.set_title("COSMOS visibility heatmap (night only, Jan–Jun 2026)")

fig.tight_layout()
fig.show()

## Polar plot

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4), subplot_kw={"projection": "polar"}, layout="constrained")

date_str = "2026-06-01 03:00:00"
observe_time = Time(date_str)


# east on the left, west on the right
ax.set_theta_direction(1)

for key, value in DEEP_FIELDS.items():
    field_name = key
    ra = value[0]
    dec = value[1]
    coords = SkyCoord(ra * u.deg, dec * u.deg, frame="icrs")
    field_target = FixedTarget(name=field_name, coord=coords)

    plot_sky(field_target, observer, observe_time, ax, style_kwargs=DEEP_FIELDS_COLORSTYLE[field_name])


ax.legend(shadow=True, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"LSST Deep Fields for day {date_str}")

plt.show()